In [15]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
knot_name = '5_2/0001.obj'
file = '../data/L400-r0.2-UpTo9Crossings/' + knot_name
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 1, [rod_radius, rod_radius])
centerline = read_nodes_from_file(file)  # supported formats: obj, txt
pr = define_periodic_rod(centerline, material)
rod_list = elastic_knots.PeriodicRodList([pr])
len(rod_list.getDoFs())

1601

In [17]:
view = Viewer(rod_list, width=1024, height=800)
view.show()


Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [7]:
def callback(problem, iteration):
    if iteration % 5 == 0:
        view.update()
for i in range(1,10):
    print(f"iterration: {i}")
    optimizerOptions = py_newton_optimizer.NewtonOptimizerOptions()
    optimizerOptions.niter = 1000
    optimizerOptions.gradTol = 1e-8
    hessianShift = 1e-4 * compute_min_eigenval_straight_rod(pr)

    problemOptions = elastic_knots.ContactProblemOptions()
    problemOptions.contactStiffness = 1e-3
    problemOptions.dHat = 2*rod_radius *i
    fixedVars = []   
    
    report = elastic_knots.compute_equilibrium(
        rod_list, problemOptions, optimizerOptions, 
        fixedVars=fixedVars,
        externalForces=np.zeros(rod_list.numDoF()),
        softConstraints=[],
        callback=callback,
        hessianShift=hessianShift
        )
    view.update()

iterration: 1
0	0.909793	0.0220395	0.0220395	0.0546875	1
1	0.907547	0.0211426	0.0211426	0.21875	1
2	0.906443	0.0194028	0.0194028	0.25	0
3	0.905495	0.0189467	0.0189467	0.09375	0
4	0.904868	0.0249486	0.0249486	0.25	1
5	0.904644	0.0194143	0.0194143	0.0234375	1
6	0.904567	0.019096	0.019096	0.046875	1
7	0.904504	0.0181955	0.0181955	0.375	1
8	0.904249	0.012717	0.012717	0.875	1
9	0.90349	0.0134592	0.0134592	0.0742188	1
10	0.903438	0.0171781	0.0171781	0.125	1
11	0.903433	0.0167531	0.0167531	1	1
12	0.903408	0.00252574	0.00252574	1	1
13	0.903384	0.00237384	0.00237384	1	1
14	0.903337	0.00414101	0.00414101	0.5	1
15	0.903299	0.0169397	0.0169397	1	1
16	0.903205	0.00346757	0.00346757	1	1
17	0.903099	0.0176345	0.0176345	0.546875	1
18	0.903015	0.0136475	0.0136475	1	1
19	0.90282	0.0055349	0.0055349	1	1
20	0.902617	0.00560129	0.00560129	1	1
21	0.902375	0.00288939	0.00288939	1	1
22	0.902015	0.00257518	0.00257518	0.0546875	0
23	0.901782	0.0055136	0.0055136	0.125	0
24	0.90144	0.00803207	0.00803207	1	1
25	0.

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	1.24012	2.18383	2.18383	0.161133	1
1	1.23474	24.3011	24.3011	0.472656	1
2	1.23287	42.225	42.225	0.625	1
3	1.23226	12.1178	12.1178	0.28125	1
4	1.23195	2.33706	2.33706	0.984375	1
5	1.23114	13.4912	13.4912	1	1
6	1.23053	1.01549	1.01549	1	1
7	1.2297	4.26517	4.26517	1	1
8	1.2288	0.984285	0.984285	1	1
9	1.22809	12.6028	12.6028	1	1
10	1.22777	1.09946	1.09946	1	1
11	1.22746	4.47855	4.47855	1	1
12	1.22706	0.221981	0.221981	1	1
13	1.22648	0.132378	0.132378	1	1
14	1.22562	0.151708	0.151708	1	1
15	1.22429	0.588238	0.588238	1	1
16	1.22233	1.66257	1.66257	1	1
17	1.21971	2.65506	2.65506	1	1
18	1.21659	0.476806	0.476806	1	1
19	1.21334	22.1347	22.1347	0.46875	1
20	1.21333	10.0111	10.0111	1	1
21	1.21331	3.6566	3.6566	1	1
22	1.21331	0.791883	0.791883	1	1
23	1.2133	0.128443	0.128443	1	1
24	1.21328	0.0332815	0.0332815	1	1
25	1.21323	0.0335898	0.0335898	1	1
26	1.21315	0.0423853	0.0423853	1	1
27	1.21298	0.0949788	0.0949788	1	1
28	1.21267	0.268727	0.268727	1	1
29	1.21213	0.690136	0.690136	1	1
30	1.21124	1.2

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	1.77725	2.29987	2.29987	1	1
1	1.73541	135.722	135.722	0.5625	1
2	1.7348	23.5649	23.5649	1	1
3	1.73477	4.21155	4.21155	1	1
4	1.73475	0.719572	0.719572	1	1
5	1.73471	0.365096	0.365096	1	1
6	1.73467	0.285696	0.285696	1	1
7	1.73462	0.211935	0.211935	1	1
8	1.73456	0.14986	0.14986	1	1
9	1.73451	0.111645	0.111645	1	1
10	1.73436	0.0917642	0.0917642	1	1
11	1.73413	0.0879241	0.0879241	1	1
12	1.73369	0.0813553	0.0813553	1	1
13	1.73289	0.123498	0.123498	1	1
14	1.73147	0.31133	0.31133	1	1
15	1.72911	0.681135	0.681135	1	1
16	1.72554	0.892336	0.892336	1	1
17	1.72071	1.13639	1.13639	1	1
18	1.71494	1.24184	1.24184	1	1
19	1.70881	1.50154	1.50154	1	1
20	1.70288	1.92497	1.92497	1	1
21	1.6975	24.926	24.926	0.476562	1
22	1.69746	3.55238	3.55238	1	1
23	1.69738	1.06924	1.06924	1	1
24	1.69724	0.0127388	0.0127388	1	1
25	1.69698	0.0562502	0.0562502	1	1
26	1.6965	0.0782233	0.0782233	1	1
27	1.69566	0.0848973	0.0848973	1	1
28	1.69432	0.0785546	0.0785546	1	1
29	1.69241	0.0739529	0.0739529	1	1
30	1.69001	0.0759513	

...

0	12.4142	4.26561	4.26561	1	1
1	12.3088	31.0575	31.0575	0.414062	1
2	12.3007	251.342	251.342	0.15625	1
3	12.2931	127.848	127.848	0.964844	1
4	12.29	187.168	187.168	1	1
5	12.2889	56.8773	56.8773	1	1
6	12.2883	9.74425	9.74425	1	1
7	12.2866	6.51748	6.51748	1	1
8	12.2842	6.17766	6.17766	1	1
9	12.2801	4.44563	4.44563	1	1
10	12.2734	1.52406	1.52406	1	1
11	12.2632	16.9461	16.9461	1	1
12	12.2495	131.769	131.769	0.000374603	1
13	12.2491	102.46	102.46	1	1
14	12.2483	22.7243	22.7243	1	1
15	12.2473	0.661661	0.661661	1	1
16	12.2436	8.16571	8.16571	1	1
17	12.2374	12.9614	12.9614	1	1
18	12.2274	2.9356	2.9356	1	1
19	12.2175	173.907	173.907	0.371094	1
20	12.213	53.6778	53.6778	1	1
21	12.2126	18.0823	18.0823	1	1
22	12.2125	1.30594	1.30594	1	1
23	12.2122	0.211757	0.211757	1	1
24	12.2117	0.268012	0.268012	1	1
25	12.2106	0.118484	0.118484	1	1
26	12.2087	1.19563	1.19563	1	1
27	12.205	4.8903	4.8903	1	1
28	12.1987	3.28057	3.28057	1	1
29	12.1987	0.212108	0.212108	1	1
30	12.1946	0.784077	0.784077	1	1
31	12.1884

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	26.0121	7.32722	7.32722	0.54043	1
1	25.937	768.287	768.287	1	1
2	25.9314	203.74	203.74	1	1
3	25.9227	41.8593	41.8593	1	1
4	25.9116	17.178	17.178	1	1
5	25.8958	57.6189	57.6189	0.0404175	1
6	25.8957	938.344	938.344	0.208496	1
7	25.8946	348.791	348.791	1	1
8	25.8928	31.5339	31.5339	1	1
9	25.8866	28.9652	28.9652	1	1
10	25.8767	23.3267	23.3267	1	1
11	25.8619	47.0171	47.0171	0.574219	1
12	25.8504	81.6227	81.6227	1	1
13	25.8473	105.513	105.513	1	1
14	25.8428	288.475	288.475	1	1
15	25.8419	24.5865	24.5865	1	1
16	25.8408	1.03022	1.03022	1	1
17	25.8387	4.12606	4.12606	1	1
18	25.8348	11.9691	11.9691	1	1
19	25.8281	20.8398	20.8398	1	1
20	25.8281	0.967099	0.967099	1	1
21	25.8281	0.229457	0.229457	1	1
22	25.828	0.272657	0.272657	1	1
23	25.8278	0.579643	0.579643	1	1
24	25.8276	0.793154	0.793154	1	1
25	25.8261	1.58988	1.58988	1	1
26	25.8048	2.46049	2.46049	1	1
27	25.7996	1.08709	1.08709	1	1
28	25.7968	0.595609	0.595609	1	1
29	25.795	0.395152	0.395152	1	1
30	25.7934	0.691499	0.691499	1	1
31	25.7913	2

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	51.0929	17.0632	17.0632	1	1
1	51.0708	105.436	105.436	1	1
2	51.0516	267.18	267.18	1	1
3	51.0329	1062.49	1062.49	1	1
4	51.0295	113.809	113.809	1	1
5	51.0255	12.5553	12.5553	1	1
6	51.019	9.43763	9.43763	1	1
7	51.0085	1.07506	1.07506	1	1
8	50.992	65.9489	65.9489	1	1
9	50.9686	85.7575	85.7575	1	1
10	50.9409	603.039	603.039	1	1
11	50.9394	208.048	208.048	1	1
12	50.9377	25.4376	25.4376	1	1
13	50.9348	7.41148	7.41148	1	1
14	50.9295	22.4348	22.4348	1	1
15	50.921	46.5314	46.5314	1	1
16	50.9087	59.8091	59.8091	1	1
17	50.8926	47.2612	47.2612	1	1
18	50.8732	123.544	123.544	1	1
19	50.8709	59.8218	59.8218	1	1
20	50.8671	196.355	196.355	1	1
21	50.8664	58.2883	58.2883	1	1
22	50.8655	2.44828	2.44828	1	1
23	50.8638	0.160972	0.160972	1	1
24	50.8593	3.95274	3.95274	1	1
25	50.8531	5.33483	5.33483	1	1
26	50.7374	14.893	14.893	1	1
27	50.6903	4.12656	4.12656	1	1
28	50.6786	1.9226	1.9226	1	1
29	50.6728	0.964841	0.964841	1	1
30	50.6695	0.595462	0.595462	1	1
31	50.6672	0.766807	0.766807	1	1
32	50.6652	1.25219	

In [8]:
from helpers import write_obj
file = '../data/NoCollision/' + knot_name
write_obj(file, rod_list)

In [13]:
# Load the centerline from file...
file = '../data/NoCollision/' + knot_name
knot = read_nodes_from_file(file)
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 0.3, [rod_radius, rod_radius])
pr = define_periodic_rod(knot[::1], material)
rod_list = elastic_knots.PeriodicRodList([pr])

In [14]:
view = Viewer(rod_list, width=1024, height=800)
view.show()

Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [ ]:
from helpers import write_obj
file = '../data/NoCollision/reduced' + knot_name
write_obj(file, rod_list)